In [1]:
!wget https://huggingface.co/datasets/barryallen16/Sarav-real-fake-automobile-parts-dataset/resolve/main/dataset1.zip

--2026-02-05 10:13:54--  https://huggingface.co/datasets/barryallen16/Sarav-real-fake-automobile-parts-dataset/resolve/main/dataset1.zip
Resolving huggingface.co (huggingface.co)... 3.170.185.35, 3.170.185.14, 3.170.185.25, ...
Connecting to huggingface.co (huggingface.co)|3.170.185.35|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/68dc24c5ee15b4bd54002f0b/ec5afaf1bdc695b853ddf32deffe40de6cbe55f48ccb0a2bc7c07690576f5fb4?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27dataset1.zip%3B+filename%3D%22dataset1.zip%22%3B&response-content-type=application%2Fzip&Expires=1770290034&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiRXBvY2hUaW1lIjoxNzcwMjkwMDM0fX0sIlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjhkYzI0YzVlZTE1YjRiZDU0MDAyZjBiL2VjNWFmYWYxYmRjNjk1Yjg1M2RkZjMyZGVmZmU0MGRlNmNiZTU1ZjQ4Y2NiMGEyYmM3YzA3NjkwNTc2ZjVmYjRcXD9yZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSomcmVz

In [2]:
!unzip dataset1.zip -d unzipped/

Archive:  dataset1.zip
   creating: unzipped/dataset1/
   creating: unzipped/dataset1/air filter fake/
  inflating: unzipped/dataset1/air filter fake/10.jpg  
  inflating: unzipped/dataset1/air filter fake/11.jpg  
  inflating: unzipped/dataset1/air filter fake/12.jpg  
  inflating: unzipped/dataset1/air filter fake/13.jpg  
  inflating: unzipped/dataset1/air filter fake/14.jpg  
  inflating: unzipped/dataset1/air filter fake/15.jpg  
  inflating: unzipped/dataset1/air filter fake/17.jpg  
  inflating: unzipped/dataset1/air filter fake/18.jpg  
  inflating: unzipped/dataset1/air filter fake/19.jpg  
  inflating: unzipped/dataset1/air filter fake/2.jpg  
  inflating: unzipped/dataset1/air filter fake/20.jpg  
  inflating: unzipped/dataset1/air filter fake/21.jpg  
  inflating: unzipped/dataset1/air filter fake/22.jpg  
  inflating: unzipped/dataset1/air filter fake/23.jpg  
  inflating: unzipped/dataset1/air filter fake/24.jpg  
  inflating: unzipped/dataset1/air filter fake/25.jpg  
  

In [34]:
import os
os.listdir("unzipped/")

['all_helmet_images', 'all_airfilter_images', 'all_sparkplug_images']

In [26]:
!mkdir unzipped/all_sparkplug_images/

In [29]:
!mv "unzipped/dataset1/spark plug fake"/* unzipped/all_sparkplug_images/

In [33]:
!rmdir "unzipped/dataset1/"

In [35]:
!pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 4.2 MB/s eta 0:00:00


In [56]:
# ==========================================
# COUNTERFEIT DETECTION V10 - VIEW-AWARE
# ==========================================

import os
import base64
import json
import glob
from groq import Groq, RateLimitError
from PIL import Image
import io
from tqdm.notebook import tqdm

# --- CONFIGURATION ---
BASE_DIR = "unzipped"
OUTPUT_FILE = "counterfeit_validation_results_v6.jsonl"
MODEL_ID = "meta-llama/llama-4-scout-17b-16e-instruct"

FOLDER_MAP = {
    "all_helmet_images": "helmet",
    "all_airfilter_images": "air_filter",
    "all_sparkplug_images": "spark_plug"
}

# ==========================================
# API KEY MANAGEMENT (same as before)
# ==========================================
def get_api_keys():
    raw_keys = ""
    try:
        from google.colab import userdata
        raw_keys = userdata.get("GROQ_API_KEYS")
    except:
        try:
            from google.colab import userdata
            raw_keys = userdata.get("GROQ_API_KEY")
        except:
            print("❌ No secrets found.")
            return []

    keys = [k.strip() for k in raw_keys.split(',') if k.strip()]
    print(f"🔑 Loaded {len(keys)} API Key(s).")
    return keys

api_keys = get_api_keys()
clients = [Groq(api_key=k) for k in api_keys]
current_key_idx = 0

def get_next_client():
    global current_key_idx
    current_key_idx = (current_key_idx + 1) % len(clients)
    return clients[current_key_idx]

# ==========================================
# V10 PROMPTS (View-Aware)
# ==========================================

PROMPTS = {
    "helmet": {
        "system": """You are a motorcycle helmet quality inspector.
CRITICAL: "Cannot see feature" ≠ "Low quality"
Rate based ONLY on what IS visible. Ignore what isn't.
""",
        "user": """
## STEP 1: CLASSIFY THE VIEW TYPE

Determine the image type:
- EXTERIOR: Outside of helmet shell visible
- INTERIOR: Inside padding/liner visible
- CLOSE_UP: Zoomed on label/strap/vent/detail
- PACKAGING: Box or bag (helmet may not be visible)
- MIXED: Multiple views

## STEP 2: VIEW-SPECIFIC ASSESSMENT

### IF EXTERIOR:
- Assess: Shell finish, symmetry, visor quality, visible branding
- Ignore: Interior padding, internal labels

### IF INTERIOR:
- Assess: Padding thickness (>25mm=good), strap rivets (metal=good), liner quality
- Ignore: Shell exterior, visor
- Thick EPS foam + metal rivets = HIGH quality interior

### IF CLOSE_UP (Label):
- Assess: Text clarity, spelling, certification marks
- Clear ISI/DOT/ECE label = POSITIVE indicator → can be HIGH quality

### IF PACKAGING:
- Assess: Print quality, brand consistency, certification claims
- Professional packaging with certs = HIGH quality packaging

## STEP 3: RATING LOGIC

⚠️ IMPORTANT:
- If visible features look GOOD → "HIGH" (even if other features not visible)
- Only rate "LOW" if you see ACTUAL defects
- If truly nothing assessable → "UNCERTAIN"

DO NOT rate "LOW" just because view is limited!

## OUTPUT JSON:
{
  "view_type": "EXTERIOR | INTERIOR | CLOSE_UP | PACKAGING | MIXED",
  "helmet_style": "FULL_FACE | OPEN_FACE | HALF_SHELL | UNKNOWN",
  "assessed_features": ["list what you COULD assess"],
  "not_in_frame": ["list what you could NOT see - DO NOT PENALIZE THESE"],
  "indicators": {
    "primary_visible_quality": "HIGH | LOW | CANNOT_ASSESS",
    "branding": "PROFESSIONAL | SUSPICIOUS | NONE_VISIBLE",
    "certifications_found": ["DOT", "ECE", "ISI", "SNELL", "CCC"] or []
  },
  "visible_defects": ["only ACTUAL defects seen"],
  "visual_quality": "HIGH | LOW | UNCERTAIN",
  "confidence": 0.0-1.0,
  "reasoning": "View type is [X]. Assessed [features]. Quality is [Y] because [Z]."
}
"""
    },

    "spark_plug": {
        "system": """You are a spark plug quality inspector.
RULE: "Not visible" ≠ "Defective". Rate ONLY what you can see.""",
        "user": """
## STEP 1: VIEW TYPE
- FULL_PRODUCT | CLOSE_UP | PACKAGING | PARTIAL

## STEP 2: ASSESS VISIBLE FEATURES
- Electrode tip: HIGH_QUALITY / LOW_QUALITY / NOT_VISIBLE
- Insulator: HIGH_QUALITY / LOW_QUALITY / NOT_VISIBLE
- Metal shell: HIGH_QUALITY / LOW_QUALITY / NOT_VISIBLE
- Branding: PROFESSIONAL / POOR / NOT_VISIBLE

## STEP 3: RATING
- Visible features good → "HIGH"
- Visible defects → "LOW"
- Can't assess enough → "UNCERTAIN"

{
  "view_type": "FULL_PRODUCT | CLOSE_UP | PACKAGING | PARTIAL",
  "brand_detected": "string or null",
  "assessed_features": ["what you could see"],
  "indicators": {
    "electrode": "HIGH_QUALITY | LOW_QUALITY | NOT_VISIBLE",
    "insulator": "HIGH_QUALITY | LOW_QUALITY | NOT_VISIBLE",
    "metal_shell": "HIGH_QUALITY | LOW_QUALITY | NOT_VISIBLE",
    "branding": "PROFESSIONAL | POOR | NOT_VISIBLE"
  },
  "visible_defects": ["actual defects only"],
  "visual_quality": "HIGH | LOW | UNCERTAIN",
  "confidence": 0.0-1.0,
  "reasoning": "Based on [visible features], quality is [X]"
}
"""
    },

    "air_filter": {
        "system": """You are an air filter quality inspector.
RULE: Rate ONLY visible features. "Not visible" ≠ "Defective".""",
        "user": """
## VIEW TYPE: FULL_PRODUCT | CLOSE_UP | PACKAGING | PARTIAL

## ASSESS VISIBLE FEATURES:
- Pleats: HIGH_QUALITY (uniform) / LOW_QUALITY (wavy) / NOT_VISIBLE
- Frame: HIGH_QUALITY (clean) / LOW_QUALITY (flash) / NOT_VISIBLE
- Seal: HIGH_QUALITY / LOW_QUALITY / NOT_VISIBLE
- Branding: PROFESSIONAL / POOR / NOT_VISIBLE

## RATING: Based on VISIBLE features only

{
  "view_type": "FULL_PRODUCT | CLOSE_UP | PACKAGING | PARTIAL",
  "brand_detected": "string or null",
  "assessed_features": ["visible items"],
  "indicators": {
    "pleat_structure": "HIGH_QUALITY | LOW_QUALITY | NOT_VISIBLE",
    "frame_housing": "HIGH_QUALITY | LOW_QUALITY | NOT_VISIBLE",
    "seal_gasket": "HIGH_QUALITY | LOW_QUALITY | NOT_VISIBLE",
    "branding": "PROFESSIONAL | POOR | NOT_VISIBLE"
  },
  "visible_defects": [],
  "visual_quality": "HIGH | LOW | UNCERTAIN",
  "confidence": 0.0-1.0,
  "reasoning": "explanation"
}
"""
    }
}

# ==========================================
# IMAGE ENCODING
# ==========================================
def encode_image(image_path, max_size_mb=3.5):
    try:
        with Image.open(image_path) as img:
            if img.mode != 'RGB': img = img.convert('RGB')
            buffer = io.BytesIO()
            img.save(buffer, format="JPEG", quality=85)
            size_mb = buffer.tell() / (1024 * 1024)
            while size_mb > max_size_mb:
                width, height = img.size
                img = img.resize((int(width * 0.75), int(height * 0.75)), Image.Resampling.LANCZOS)
                buffer = io.BytesIO()
                img.save(buffer, format="JPEG", quality=85)
                size_mb = buffer.tell() / (1024 * 1024)
            return base64.b64encode(buffer.getvalue()).decode('utf-8')
    except Exception as e:
        print(f"⚠️ Image Error {image_path}: {e}")
        return None

# ==========================================
# MAIN EXECUTION
# ==========================================
tasks = []
for folder_name, prompt_key in FOLDER_MAP.items():
    full_path = os.path.join(BASE_DIR, folder_name)
    if os.path.exists(full_path):
        files = glob.glob(os.path.join(full_path, "*.*"))
        images = [f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))]
        for img in images:
            tasks.append({"path": img, "type": prompt_key})

print(f"🚀 Processing {len(tasks)} images with V10 prompts...")


🔑 Loaded 6 API Key(s).
🚀 Processing 274 images with V10 prompts...


In [57]:
processed_imgpaths=set()
with open('./counterfeit_validation_results_v5.jsonl', 'r', encoding='utf-8') as in_file:
  for line in in_file:
    data = json.loads(line)
    image_path = data["folder"] +"/"+ data['filename']
    processed_imgpaths.add(image_path)

In [58]:
with open(OUTPUT_FILE, "a") as f_out:
    for task in tqdm(tasks):
        img_path = task["path"]
        if img_path in processed_imgpaths: continue
        part_type = task["type"]
        filename = os.path.basename(img_path)
        prompt_data = PROMPTS[part_type]

        base64_image = encode_image(img_path)
        if not base64_image: continue

        success = False
        attempts = 0
        max_attempts = len(clients) * 2

        while not success and attempts < max_attempts:
            client = clients[current_key_idx]
            try:
                completion = client.chat.completions.create(
                    model=MODEL_ID,
                    messages=[
                        {"role": "system", "content": prompt_data["system"]},
                        {
                            "role": "user",
                            "content": [
                                {"type": "text", "text": prompt_data["user"]},
                                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}}
                            ]
                        }
                    ],
                    temperature=0.1,
                    response_format={"type": "json_object"},
                    stream=False
                )

                result = {
                    "filename": filename,
                    "folder": os.path.dirname(img_path),
                    "part_type": part_type,
                    "prompt_version": "v10",
                    "analysis": json.loads(completion.choices[0].message.content)
                }
                f_out.write(json.dumps(result) + "\n")
                f_out.flush()
                success = True

            except RateLimitError:
                client = get_next_client()
                attempts += 1
            except Exception as e:
                print(f"❌ Error on {filename}: {str(e)}")
                break

print(f"✅ Done. Results saved to {OUTPUT_FILE}")

  0%|          | 0/274 [00:00<?, ?it/s]

❌ Error on 3.jpg: Error code: 400 - {'error': {'message': "'messages' must contain the word 'json' in some form, to use 'response_format' of type 'json_object'.", 'type': 'invalid_request_error'}}
❌ Error on 57.jpg: Error code: 400 - {'error': {'message': "'messages' must contain the word 'json' in some form, to use 'response_format' of type 'json_object'.", 'type': 'invalid_request_error'}}
❌ Error on 80.jpg: Error code: 400 - {'error': {'message': "'messages' must contain the word 'json' in some form, to use 'response_format' of type 'json_object'.", 'type': 'invalid_request_error'}}
❌ Error on 42.jpg: Error code: 400 - {'error': {'message': "'messages' must contain the word 'json' in some form, to use 'response_format' of type 'json_object'.", 'type': 'invalid_request_error'}}
❌ Error on 45.jpg: Error code: 400 - {'error': {'message': "'messages' must contain the word 'json' in some form, to use 'response_format' of type 'json_object'.", 'type': 'invalid_request_error'}}
❌ Error on 

In [55]:
import os

base_directory = 'unzipped'
file_count = 0

for root, dirs, files in os.walk(base_directory):
    file_count += len(files)

print(f"Total number of files in '{base_directory}' and its subdirectories: {file_count}")

Total number of files in 'unzipped' and its subdirectories: 277


In [41]:
!pip install -q groq

import os
import base64
import json
import io
from PIL import Image
from groq import Groq

# --- CONFIGURATION ---
# 1. Paste ONE of your API keys here for the test
TEST_API_KEY = os.getenv("GROQ_API_KEY", "")

# 2. Pick a test image path from your unzipped folder
# Example: "unzipped/all_helmet_images/helmet_123.jpg"
TEST_IMAGE_PATH = "unzipped/all_sparkplug_images/1.jpg"

# 3. Choose the part type for the test: "spark_plug", "helmet", or "air_filter"
TEST_PART_TYPE = "spark_plug"
# ---------------------

# Setup Client
client = Groq(api_key=TEST_API_KEY)
MODEL_ID = "meta-llama/llama-4-scout-17b-16e-instruct"

# Verified Prompt (Spark Plug Example - matches v5)
PROMPTS = {
    "spark_plug": {
        "system": "You are an expert Quality Control Inspector trained by Niterra (NGK/NTK). Your job is to identify counterfeit spark plugs using visual defects.",
        "user": """Analyze this image for the 5 OFFICIAL NGK Anti-Counterfeit checks. Output strictly in JSON:
        1. LOT CODE: Is a 4-digit alphanumeric code stamped on the hex? (Pass/Fail)
        2. CAPTIVE GASKET: Is the metal washer loose/threaded down (Fail) or captive (Pass)?
        3. MACHINING MARKS: Are there horizontal lathe lines on the C-Groove or Crimping? (Fail)
        4. ELECTRODE TIP: Is the tip fine-wire Iridium/Platinum (Pass) or thick/blunt Nickel (Fail)?
        5. BRANDING QUALITY: Is the logo crisp and centered (Pass) or smudged/off-center (Fail)?

        Output format: {"classification": "GENUINE|COUNTERFEIT|UNCERTAIN", "confidence": 0.0-1.0, "primary_reason": "string", "checks": {"lot_code": "pass/fail", "gasket": "pass/fail", "machining": "pass/fail", "electrode": "pass/fail", "branding": "pass/fail"}}"""
    }
    # (Other prompts omitted for brevity in test)
}

def encode_image(image_path):
    with Image.open(image_path) as img:
        if img.mode != 'RGB': img = img.convert('RGB')
        buffer = io.BytesIO()
        img.save(buffer, format="JPEG", quality=85)
        return base64.b64encode(buffer.getvalue()).decode('utf-8')

print(f"🧪 Testing on: {TEST_IMAGE_PATH}")

try:
    # Prepare
    base64_image = encode_image(TEST_IMAGE_PATH)
    prompt_data = PROMPTS[TEST_PART_TYPE]

    # Call API
    completion = client.chat.completions.create(
        model=MODEL_ID,
        messages=[
            {"role": "system", "content": prompt_data["system"]},
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt_data["user"]},
                    {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}}
                ]
            }
        ],
        temperature=0.1,
        response_format={"type": "json_object"}
    )

    # Print Result
    print("\n✅ API RESPONSE:")
    print(json.dumps(json.loads(completion.choices[0].message.content), indent=2))

except Exception as e:
    print(f"\n❌ TEST FAILED: {e}")

🧪 Testing on: unzipped/all_sparkplug_images/1.jpg

✅ API RESPONSE:
{
  "classification": "COUNTERFEIT",
  "confidence": 1.0,
  "primary_reason": "The spark plug in the image is branded as BOSCH, not NGK, and does not match NGK's design.",
  "checks": {
    "lot_code": "fail",
    "gasket": "fail",
    "machining": "fail",
    "electrode": "fail",
    "branding": "fail"
  }
}


In [85]:
# ==========================================
# COUNTERFEIT DETECTION V10 - OPENROUTER
# ==========================================

import os
import base64
import json
import glob
import time
from openai import OpenAI
from PIL import Image
import io
from tqdm.notebook import tqdm

# --- CONFIGURATION ---
BASE_DIR = "unzippp"
OUTPUT_FILE = "counterfeit_validation_results_v8_openrouter.jsonl"

# OpenRouter model options (pick one)
MODEL_ID = "nvidia/nemotron-nano-12b-v2-vl:free"  # Free tier
# MODEL_ID = "meta-llama/llama-4-maverick:free"  # Alternative free
# MODEL_ID = "google/gemini-2.0-flash-001"  # Paid but cheap
# MODEL_ID = "anthropic/claude-3.5-sonnet"  # Higher quality, paid

FOLDER_MAP = {
    "all_helmet_images": "helmet",
    "all_airfilter_images": "air_filter",
    "all_sparkplug_images": "spark_plug"
}

# ==========================================
# OPENROUTER API KEY MANAGEMENT
# ==========================================

def get_api_keys():
    """Retrieves OpenRouter keys from Colab/Kaggle secrets."""
    raw_keys = ""

    # Try Colab secrets first
    try:
        from google.colab import userdata
        raw_keys = userdata.get("OPENROUTER_APIKEYS")
        if not raw_keys:
            raw_keys = userdata.get("OPENROUTER_API_KEY")
    except:
        pass

    # Try environment variable
    if not raw_keys:
        raw_keys = os.environ.get("OPENROUTER_API_KEYS", "")
        if not raw_keys:
            raw_keys = os.environ.get("OPENROUTER_API_KEY", "")

    if not raw_keys:
        print("❌ No OpenRouter API keys found!")
        print("   Set OPENROUTER_API_KEYS in Colab secrets or environment")
        return []

    keys = [k.strip() for k in raw_keys.split(',') if k.strip()]
    print(f"🔑 Loaded {len(keys)} OpenRouter API Key(s)")
    return keys

def create_openrouter_client(api_key):
    """Create OpenRouter client using OpenAI SDK."""
    return OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=api_key,
        default_headers={
            "HTTP-Referer": "https://github.com/counterfeit-detection",  # Optional
            "X-Title": "Counterfeit Parts Detection Pipeline"  # Optional
        }
    )

# Initialize clients
api_keys = get_api_keys()
clients = [create_openrouter_client(k) for k in api_keys]
current_key_idx = 0

def get_next_client():
    """Rotate to next available client."""
    global current_key_idx
    current_key_idx = (current_key_idx + 1) % len(clients)
    print(f"   🔄 Rotated to key {current_key_idx + 1}/{len(clients)}")
    return clients[current_key_idx]

# ==========================================
# V10 PROMPTS (View-Aware) - UNCHANGED
# ==========================================

PROMPTS = {
    "helmet": {
        "system": """You are a motorcycle helmet quality inspector.
CRITICAL: "Cannot see feature" ≠ "Low quality"
Rate based ONLY on what IS visible. Ignore what isn't.
""",
        "user": """
## STEP 1: CLASSIFY THE VIEW TYPE

Determine the image type:
- EXTERIOR: Outside of helmet shell visible
- INTERIOR: Inside padding/liner visible
- CLOSE_UP: Zoomed on label/strap/vent/detail
- PACKAGING: Box or bag (helmet may not be visible)
- MIXED: Multiple views

## STEP 2: VIEW-SPECIFIC ASSESSMENT

### IF EXTERIOR:
- Assess: Shell finish, symmetry, visor quality, visible branding
- Ignore: Interior padding, internal labels

### IF INTERIOR:
- Assess: Padding thickness (>25mm=good), strap rivets (metal=good), liner quality
- Ignore: Shell exterior, visor
- Thick EPS foam + metal rivets = HIGH quality interior

### IF CLOSE_UP (Label):
- Assess: Text clarity, spelling, certification marks
- Clear ISI/DOT/ECE label = POSITIVE indicator → can be HIGH quality

### IF PACKAGING:
- Assess: Print quality, brand consistency, certification claims
- Professional packaging with certs = HIGH quality packaging

## STEP 3: RATING LOGIC

⚠️ IMPORTANT:
- If visible features look GOOD → "HIGH" (even if other features not visible)
- Only rate "LOW" if you see ACTUAL defects
- If truly nothing assessable → "UNCERTAIN"

DO NOT rate "LOW" just because view is limited!

## OUTPUT JSON:
{
  "view_type": "EXTERIOR | INTERIOR | CLOSE_UP | PACKAGING | MIXED",
  "helmet_style": "FULL_FACE | OPEN_FACE | HALF_SHELL | UNKNOWN",
  "assessed_features": ["list what you COULD assess"],
  "not_in_frame": ["list what you could NOT see - DO NOT PENALIZE THESE"],
  "indicators": {
    "primary_visible_quality": "HIGH | LOW | CANNOT_ASSESS",
    "branding": "PROFESSIONAL | SUSPICIOUS | NONE_VISIBLE",
    "certifications_found": ["DOT", "ECE", "ISI", "SNELL", "CCC"] or []
  },
  "visible_defects": ["only ACTUAL defects seen"],
  "visual_quality": "HIGH | LOW | UNCERTAIN",
  "confidence": 0.0-1.0,
  "reasoning": "View type is [X]. Assessed [features]. Quality is [Y] because [Z]."
}
"""
    },

    "spark_plug": {
        "system": """You are a spark plug quality inspector.
RULE: "Not visible" ≠ "Defective". Rate ONLY what you can see.""",
        "user": """
## STEP 1: VIEW TYPE
- FULL_PRODUCT | CLOSE_UP | PACKAGING | PARTIAL

## STEP 2: ASSESS VISIBLE FEATURES
- Electrode tip: HIGH_QUALITY / LOW_QUALITY / NOT_VISIBLE
- Insulator: HIGH_QUALITY / LOW_QUALITY / NOT_VISIBLE
- Metal shell: HIGH_QUALITY / LOW_QUALITY / NOT_VISIBLE
- Branding: PROFESSIONAL / POOR / NOT_VISIBLE

## STEP 3: RATING
- Visible features good → "HIGH"
- Visible defects → "LOW"
- Can't assess enough → "UNCERTAIN"

{
  "view_type": "FULL_PRODUCT | CLOSE_UP | PACKAGING | PARTIAL",
  "brand_detected": "string or null",
  "assessed_features": ["what you could see"],
  "indicators": {
    "electrode": "HIGH_QUALITY | LOW_QUALITY | NOT_VISIBLE",
    "insulator": "HIGH_QUALITY | LOW_QUALITY | NOT_VISIBLE",
    "metal_shell": "HIGH_QUALITY | LOW_QUALITY | NOT_VISIBLE",
    "branding": "PROFESSIONAL | POOR | NOT_VISIBLE"
  },
  "visible_defects": ["actual defects only"],
  "visual_quality": "HIGH | LOW | UNCERTAIN",
  "confidence": 0.0-1.0,
  "reasoning": "Based on [visible features], quality is [X]"
}
"""
    },

    "air_filter": {
        "system": """You are an air filter quality inspector.
RULE: Rate ONLY visible features. "Not visible" ≠ "Defective".""",
        "user": """
## VIEW TYPE: FULL_PRODUCT | CLOSE_UP | PACKAGING | PARTIAL

## ASSESS VISIBLE FEATURES:
- Pleats: HIGH_QUALITY (uniform) / LOW_QUALITY (wavy) / NOT_VISIBLE
- Frame: HIGH_QUALITY (clean) / LOW_QUALITY (flash) / NOT_VISIBLE
- Seal: HIGH_QUALITY / LOW_QUALITY / NOT_VISIBLE
- Branding: PROFESSIONAL / POOR / NOT_VISIBLE

## RATING: Based on VISIBLE features only

{
  "view_type": "FULL_PRODUCT | CLOSE_UP | PACKAGING | PARTIAL",
  "brand_detected": "string or null",
  "assessed_features": ["visible items"],
  "indicators": {
    "pleat_structure": "HIGH_QUALITY | LOW_QUALITY | NOT_VISIBLE",
    "frame_housing": "HIGH_QUALITY | LOW_QUALITY | NOT_VISIBLE",
    "seal_gasket": "HIGH_QUALITY | LOW_QUALITY | NOT_VISIBLE",
    "branding": "PROFESSIONAL | POOR | NOT_VISIBLE"
  },
  "visible_defects": [],
  "visual_quality": "HIGH | LOW | UNCERTAIN",
  "confidence": 0.0-1.0,
  "reasoning": "explanation"
}
"""
    }
}

# ==========================================
# IMAGE ENCODING
# ==========================================

def encode_image(image_path, max_size_mb=3.5):
    """Resize and base64 encode image for API."""
    try:
        with Image.open(image_path) as img:
            # Convert to RGB if needed
            if img.mode != 'RGB':
                img = img.convert('RGB')

            # Initial encode
            buffer = io.BytesIO()
            img.save(buffer, format="JPEG", quality=85)
            size_mb = buffer.tell() / (1024 * 1024)

            # Resize if too large
            while size_mb > max_size_mb:
                width, height = img.size
                img = img.resize(
                    (int(width * 0.75), int(height * 0.75)),
                    Image.Resampling.LANCZOS
                )
                buffer = io.BytesIO()
                img.save(buffer, format="JPEG", quality=85)
                size_mb = buffer.tell() / (1024 * 1024)

            return base64.b64encode(buffer.getvalue()).decode('utf-8')

    except Exception as e:
        print(f"⚠️ Image Error {image_path}: {e}")
        return None

# ==========================================
# OPENROUTER API CALL WITH RETRY
# ==========================================

def analyze_image(image_path, part_type, max_retries=None):
    """
    Analyze single image via OpenRouter API with key rotation.
    """
    global current_key_idx

    if max_retries is None:
        max_retries = len(clients) * 2

    prompt_data = PROMPTS[part_type]
    base64_image = encode_image(image_path)

    if not base64_image:
        return None

    attempts = 0
    last_error = None

    while attempts < max_retries:
        client = clients[current_key_idx]

        try:
            completion = client.chat.completions.create(
                model=MODEL_ID,
                messages=[
                    {
                        "role": "system",
                        "content": prompt_data["system"]
                    },
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "text",
                                "text": prompt_data["user"]
                            },
                            {
                                "type": "image_url",
                                "image_url": {
                                    "url": f"data:image/jpeg;base64,{base64_image}"
                                }
                            }
                        ]
                    }
                ],
                temperature=0.1,
                max_tokens=1024,
                response_format={"type": "json_object"}
            )

            # Parse response
            response_text = completion.choices[0].message.content

            # Handle potential JSON parsing issues
            try:
                analysis = json.loads(response_text)
            except json.JSONDecodeError:
                # Try to extract JSON from response
                import re
                json_match = re.search(r'\{.*\}', response_text, re.DOTALL)
                if json_match:
                    analysis = json.loads(json_match.group())
                else:
                    raise ValueError("Could not parse JSON from response")

            return {
                "filename": os.path.basename(image_path),
                "folder": os.path.dirname(image_path),
                "part_type": part_type,
                "prompt_version": "v10",
                "model": MODEL_ID,
                "analysis": analysis
            }

        except Exception as e:
            error_str = str(e).lower()
            last_error = e

            # Rate limit - rotate key and retry
            if any(x in error_str for x in ['429', 'rate limit', 'rate_limit', 'too many']):
                get_next_client()
                attempts += 1
                time.sleep(1)  # Brief pause
                continue

            # Quota exceeded - rotate key
            elif any(x in error_str for x in ['quota', 'exceeded', 'insufficient']):
                print(f"   ⚠️ Key {current_key_idx + 1} quota exceeded")
                get_next_client()
                attempts += 1
                continue

            # Server error - retry with backoff
            elif any(x in error_str for x in ['500', '502', '503', 'server error']):
                wait_time = 2 ** attempts
                print(f"   ⚠️ Server error, waiting {wait_time}s...")
                time.sleep(wait_time)
                attempts += 1
                continue

            # Other error - log and break
            else:
                print(f"❌ Error on {os.path.basename(image_path)}: {e}")
                break

    # All retries failed
    return {
        "filename": os.path.basename(image_path),
        "folder": os.path.dirname(image_path),
        "part_type": part_type,
        "error": str(last_error)
    }

# ==========================================
# LOAD EXISTING PROGRESS
# ==========================================

def load_processed_files(output_file):
    """Load already processed filenames to support resume."""
    processed = set()
    if os.path.exists(output_file):
        with open(output_file, 'r') as f:
            for line in f:
                if line.strip():
                    try:
                        data = json.loads(line)
                        processed.add(data.get('filename', ''))
                    except:
                        continue
    return processed

# ==========================================
# MAIN EXECUTION
# ==========================================

def main():
    # Gather all image tasks
    tasks = []
    for folder_name, prompt_key in FOLDER_MAP.items():
        full_path = os.path.join(BASE_DIR, folder_name)
        if os.path.exists(full_path):
            files = glob.glob(os.path.join(full_path, "*.*"))
            images = [f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))]
            for img in images:
                tasks.append({"path": img, "type": prompt_key})
            print(f"📁 {folder_name}: {len(images)} images")
        else:
            print(f"⚠️ Folder not found: {full_path}")

    print(f"\n🚀 Total: {len(tasks)} images to process")
    print(f"🤖 Model: {MODEL_ID}")
    print(f"🔑 API Keys: {len(clients)}")

    # Load already processed (for resume support)
    processed = load_processed_files(OUTPUT_FILE)
    if processed:
        print(f"⏩ Resuming: {len(processed)} already done")
        tasks = [t for t in tasks if os.path.basename(t['path']) not in processed]
        print(f"📋 Remaining: {len(tasks)} images")

    if not tasks:
        print("✅ All images already processed!")
        return

    # Process images
    success_count = 0
    error_count = 0

    with open(OUTPUT_FILE, "a") as f_out:
        for task in tqdm(tasks, desc="Processing"):
            result = analyze_image(task["path"], task["type"])

            if result:
                if "error" not in result:
                    success_count += 1
                else:
                    error_count += 1

                f_out.write(json.dumps(result) + "\n")
                f_out.flush()

            # Small delay to be nice to the API
            time.sleep(0.1)

    # Summary
    print("\n" + "=" * 50)
    print("📊 PROCESSING COMPLETE")
    print("=" * 50)
    print(f"✅ Successful: {success_count}")
    print(f"❌ Errors: {error_count}")
    print(f"💾 Results: {OUTPUT_FILE}")

# Run
if __name__ == "__main__":
    main()

🔑 Loaded 14 OpenRouter API Key(s)
📁 all_helmet_images: 158 images
📁 all_airfilter_images: 117 images
📁 all_sparkplug_images: 169 images

🚀 Total: 444 images to process
🤖 Model: nvidia/nemotron-nano-12b-v2-vl:free
🔑 API Keys: 14
⏩ Resuming: 444 already done
📋 Remaining: 0 images
✅ All images already processed!


In [ ]:
os.listdir('./unzippp/dataset1/air filter fake/')

In [71]:
import os

directory_path = './unzippp/dataset1/spark plug og'

# Get a list of all files in the directory
files = os.listdir(directory_path)

for filename in files:
    # Construct the old and new file paths
    old_file_path = os.path.join(directory_path, filename)
    new_filename = 'spo_' + filename
    new_file_path = os.path.join(directory_path, new_filename)

    # Rename the file
    os.rename(old_file_path, new_file_path)
    print(f"Renamed '{filename}' to '{new_filename}'")

print("All files renamed successfully!")

Renamed '3.jpg' to 'spo_3.jpg'
Renamed '57.jpg' to 'spo_57.jpg'
Renamed '97.jpg' to 'spo_97.jpg'
Renamed '80.jpg' to 'spo_80.jpg'
Renamed '61TF4nwfzzL._SX522_.jpg' to 'spo_61TF4nwfzzL._SX522_.jpg'
Renamed '103.jpg' to 'spo_103.jpg'
Renamed '42.jpg' to 'spo_42.jpg'
Renamed '45.jpg' to 'spo_45.jpg'
Renamed '101.jpg' to 'spo_101.jpg'
Renamed '38.jpg' to 'spo_38.jpg'
Renamed '66.jpg' to 'spo_66.jpg'
Renamed '52.jpg' to 'spo_52.jpg'
Renamed '69.jpg' to 'spo_69.jpg'
Renamed '30.jpg' to 'spo_30.jpg'
Renamed '35.jpg' to 'spo_35.jpg'
Renamed '86.jpg' to 'spo_86.jpg'
Renamed '74.jpg' to 'spo_74.jpg'
Renamed '83.jpg' to 'spo_83.jpg'
Renamed '14.jpg' to 'spo_14.jpg'
Renamed '8.jpg' to 'spo_8.jpg'
Renamed '68.jpg' to 'spo_68.jpg'
Renamed '73' to 'spo_73'
Renamed '11.jpg' to 'spo_11.jpg'
Renamed '61.jpg' to 'spo_61.jpg'
Renamed '75.jpg' to 'spo_75.jpg'
Renamed '81.jpg' to 'spo_81.jpg'
Renamed '53.jpg' to 'spo_53.jpg'
Renamed '76.jpg' to 'spo_76.jpg'
Renamed '28.jpg' to 'spo_28.jpg'
Renamed '27.jpg' 

In [78]:
!mkdir ./unzippp/all_sparkplug_images

In [81]:
!mv "./unzippp/dataset1/spark plug og"/*  ./unzippp/all_sparkplug_images

In [82]:
!rm -rf ./unzippp/dataset1/